In [1]:
# %%
# Install dependencies if needed
# !pip install fairlearn matplotlib scikit-learn

# %%
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from sklearn.datasets import fetch_openml
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split

from fairlearn.reductions import GridSearch, DemographicParity
from fairlearn.metrics import MetricFrame, selection_rate, accuracy_score

# %%
# Load the Adult Census dataset from OpenML
# Predict if income >50K, sensitive feature = sex
adult = fetch_openml(data_id=1590, as_frame=True)
X = adult.data
y = adult.target

# Sensitive feature (protected attribute)
sensitive_feature = X['sex']

# Drop unneeded columns
X = X.drop(columns=['sex'])

# Train/test split
X_train, X_test, y_train, y_test, sf_train, sf_test = train_test_split(
    X, y, sensitive_feature, test_size=0.3, random_state=42, stratify=y
)

# %%
# Baseline model: Logistic Regression
baseline = LogisticRegression(solver="liblinear")
baseline.fit(X_train, y_train)

baseline_preds = baseline.predict(X_test)

# Evaluate baseline fairness
mf_baseline = MetricFrame(
    metrics={
        'accuracy': accuracy_score,
        'selection_rate': selection_rate,
    },
    y_true=y_test,
    y_pred=baseline_preds,
    sensitive_features=sf_test
)

print("=== Baseline Model ===")
print("Overall Accuracy:", accuracy_score(y_test, baseline_preds))
print("Group metrics:\n", mf_baseline.by_group)

# %%
# Fairlearn Mitigation with GridSearch under Demographic Parity
constraint = DemographicParity()
mitigator = GridSearch(
    estimator=LogisticRegression(solver="liblinear"),
    constraints=constraint,
    grid_size=10
)

mitigator.fit(X_train, y_train, sensitive_features=sf_train)

# Get all candidate models from grid search
predictors = mitigator.predictors_

# %%
# Evaluate each model for trade-offs
results = []
for i, predictor in enumerate(predictors):
    preds = predictor.predict(X_test)
    acc = accuracy_score(y_test, preds)
    mf = MetricFrame(metrics=selection_rate, y_true=y_test, y_pred=preds, sensitive_features=sf_test)
    disparity = mf.difference()
    results.append((i, acc, disparity))

results_df = pd.DataFrame(results, columns=["Model", "Accuracy", "DemographicParityDiff"])
print(results_df)

# %%
# Plot Accuracy vs Demographic Parity Difference
plt.figure(figsize=(6,4))
plt.scatter(results_df["Accuracy"], results_df["DemographicParityDiff"], c=results_df["Model"], cmap="viridis")
plt.xlabel("Accuracy")
plt.ylabel("Demographic Parity Difference")
plt.title("Fairness-Accuracy Trade-off (GridSearch)")
plt.colorbar(label="Model Index")
plt.show()


ModuleNotFoundError: No module named 'fairlearn'